<span style="font-family: 'Georgia'; font-size: 15px;">
    
## <span style="color:green"> Summary of Chatbot development: </span>

You will develop a data-driven **chatbot** in 4 stages. 
    
Each stage covers a core chatbot functionality - conversation loop, pattern matching, data-driven responses from files, and pre-processing with substitutions. 
    
This development will cover all learning materials from **weeks 1-10**. 

Please ensure that you learn respective weeks content before attempting different parts. Each stage will expand on previous stage by incorporating a new concept about data programming that you have learnt.

The following key concepts will be covered in each stage of chatbot development. 

- **Part 1.** Conversation loop
- **Part 2.** Pattern matching
- **Part 3.** File handling
- **Part 4.** NLTK and Word substitutions

<div style="text-align: center;">  <span style="color:red; font-size: 18px;"> <i> Please do these activities in a sequential order, starting from Part 1 to ending in Part 4. </i> </span> </div>



</span>


<hr style="border: 2px solid green;">

<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 1 </span>
    
**Conversation loop**

<!-- #### <span style="color:green"> 1. Chatbot with lists, conditions and string concatenation </span>
 -->

You will build the first part of the chatbot that responds to a fixed set of inputs using **lists** and **conditions**.

In order to complete this activity, please get yourself familiar with concepts covered from **Weeks 1 to Week 4** of the video material. 

**Learning outcomes**
- Creating a chatbot conversational loop with lists and conditonals
- String concatenation

Complete this activity before proceeding to the next parts of the chatbot.

</span>


---

Please review the following conversation. Your chatbot should demonstrate a behavior similar to this.


#### Sample conversation snippet:

> User: Hello

> Chatbot: Hello! How can I assist you today?

> User: I'd like to have some tea

> Chatbot: Sure, where would you like to have it? A cafe in the campus maybe?

> User: Yes, a cafe.

> Chatbot: There are 2 cafes - in main building and near the lecture halls.

> User: Thank you.

> Chatbot: You are welcome.

> User: Bye

-- Exits the chat --  

---

In [9]:
import json
import re
import random
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Download required NLTK resources (only needs to be run once)
print("Downloading NLTK data...");
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')
print("Downloads complete!")

[nltk_data] Downloading package punkt to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...


Downloads complete!


[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
def load_chatbot_data(intents_path='intents.json', synonyms_path='synonyms.json'):
    # Load Intents
    with open(intents_path, 'r') as file:
        data = json.load(file)
        
    rgx2int = {}
    int2res = {}
    
    for item in data['intents']:
        # Part 3.2: Convert strings to regex by joining patterns with the '|' (OR) operator
        combined_pattern = r"|".join(item['patterns'])
        
        # Compile regex for efficiency and case-insensitivity
        compiled_regex = re.compile(combined_pattern, re.IGNORECASE)
        
        rgx2int[compiled_regex] = item['intent']
        int2res[item['intent']] = item['responses']
        
    # Load Synonyms
    with open(synonyms_path, 'r') as file:
        synonyms_dict = json.load(file)
        
    return rgx2int, int2res, synonyms_dict

In [3]:
def preprocess_input(user_input, synonyms_dict):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(user_input)
    
    processed_tokens = []
    for word in tokens:
        word_lower = word.lower()
        
        # 1. Synonym Substitution
        if word_lower in synonyms_dict:
            word_lower = synonyms_dict[word_lower]
            
        # 2. Lemmatization (e.g., "running" -> "run", "better" -> "good")
        # Note: 'v' denotes verb, 'a' denotes adjective
        lemmatized_word = lemmatizer.lemmatize(word_lower, pos='v') 
        lemmatized_word = lemmatizer.lemmatize(lemmatized_word, pos='a')
        
        processed_tokens.append(lemmatized_word)
        
    return " ".join(processed_tokens)

In [4]:
def detect_pattern(processed_input, raw_input, rgx2int, user_state):
    for pattern, intent in rgx2int.items():
        # We search against the RAW input to safely capture proper nouns (names/colors)
        # However, for pure intent matching without variables, you can search against processed_input
        match = pattern.search(raw_input) 
        
        if not match:
            # Fallback: check processed input in case lemmatization is needed to trigger the match
            match = pattern.search(processed_input)
            
        if match:
            # If the regex uses named groups (?P<name>...), extract and store them in memory
            if match.groupdict():
                user_state.update(match.groupdict())
            return intent
            
    return "unknown"

def chatbot_response(intent, int2res, user_state):
    if intent in int2res:
        # Choose a random response from the available list
        selected_response = random.choice(int2res[intent])
        
        # Apply dynamic string substitution (Part 2.6)
        try:
            return selected_response.format(**user_state)
        except KeyError:
            # Failsafe if the template requires a variable not yet in memory
            return selected_response
    else:
        return "I'm not quite sure I understand. Could you rephrase that?"

In [ ]:
def chatbot():
    print("Welcome! Type 'quit', 'exit', or 'bye' to exit the chat.\n")
    
    # 1. Load data from files
    rgx2int, int2res, synonyms_dict = load_chatbot_data()
    
    # 2. Initialize user state memory
    user_state = {}
    
    # ADVANCED FEATURE: Sentiment Analysis List
    negative_words = ["stressed", "confused", "frustrated", "lost", "hard", "difficult", "overwhelmed", "stuck"]

    while True:
        raw_input = input("You: ").strip()
        
        # Immediate termination check
        if raw_input.lower() in ["bye", "quit", "exit"]:
            print("Chatbot: Thank you for your interaction. Goodbye!")
            break
            
        # 3. Preprocess user input (Tokenize, Synonyms, Lemmatize)
        processed_input = preprocess_input(raw_input, synonyms_dict)
        
        # --- ADVANCED FEATURE INTERCEPT ---
        # Check if any word in the processed input matches our negative sentiment list
        is_stressed = any(word in processed_input.split() for word in negative_words)
        
        if is_stressed:
            # Bypass normal intent matching and pull from the specific empathetic intent
            print("Chatbot: I know navigating university requirements can be overwhelming. Take a deep breath! Which portal or document is causing trouble?")
            continue # Skips the rest of the loop and waits for their next input
        # 4. Detect Intent & Update Memory
        intent = detect_pattern(processed_input, raw_input, rgx2int, user_state)
        
        # 5. Generate and print response
        response = chatbot_response(intent, int2res, user_state)
        
        print(f"Chatbot: {response}")

# Run the chatbot
chatbot()

Welcome! Type 'quit', 'exit', or 'bye' to exit the chat.

Chatbot: Hello! How can I help you today?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: Hello gg, how can I assist you?
Chatbot: Hello GG, how can I assist you?
Chatbot: Thank you for your interaction. Goodbye!



<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 1 </span> </div>
<hr style="border: 2px solid green;">


<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 2 </span>
**Pattern matching**


<!-- #### <span style="color:green"> 2. Chatbot with Dictionaries, and regular expressions </span> -->

You will enhance the basic chatbot to recognize patterns using **regular expressions** and organize code as **functions**.
        
Before proceeding, familiarise yourself with **Week 5 and 6** of lecture material on **Dictionaries** and **Regular expressions**.

<span style="color:red"> *Before you start this, please ensure that you complete all the previous parts (Part 1).* </span>


**Learning outcomes**
- Writing and storing patterns (2.1, 2.2). 
- Functions for pattern matching, and retrieval (2.3, 2.4). 
- String substitution (2.5, 2.6)

Complete this activity before proceeding to the next parts of the chatbot.

</span>


---

*<span style="color:red; font-size: 15px"> Modular programming: </span> We will be experimenting with several different types of pattern matching techniques for chatbot. As we do not want to end up with stray code for chatbot versions, for each new chatbot feature, we will write a function and call the functions in the main. At the end, we will have multiple functions for pattern matching and response generation, and we will be be in a position to mix and match them in the main loop for comparison.*

---

## Writing and storing patterns

You will write two functions to create dictionaries and store patterns in them. 

- First, you will use the ZIP function to combine two independent lists (precepts and Chatbot responses) 


- Then, you will create 5 regex patterns and store them in a dictionary named "re_patterns"


### 2.1 Lists -> Dictionaries

To convert lists to dictionaries, let's use the lists from previous activity.

Instead of writing a new *dictionary*, create this using the **ZIP** function on two lists. 

In [ ]:
# Write a function that takes in precept categories and responses lists and returns a mapping between them.

# - Use ZIP function to combine two independent lists (precepts and responses) and create a new map between them.
# - Search on the ZIP function and how to use it in a for loop
# - Run the main loop and check that this function works

# ___ Write your code here _____

def zip_responses(precepts, response):

    mapping = {}
    return mapping

Use zip function to create a dictionary with 5 precepts-responses pairs. 

Use a list to accomodate more than one response. 

Example dictionary:
    
    {
        "greetings": ["how are you?"],
        "tea": ["I love this beverage", "Can you tell me more?, etc... "]
         ...
    }
    
   

In [ ]:
# Dictionary name : "responses" 

# ___ Write your code here _____

---

### 2.2 Regular expression -> Patterns -> Dictionary keys.

- Match complete words using regular expressions. 
- Example, match all occurrences of hello, hi, hey etc under a pattern called greetings.
- Use "|" symbol to string together multiple words.
- Each pattern should have atleast or 5 different occurrences of the word you want to match.
- Create atleast 5 different patterns.

Example dictionary:
    
    {"hi|hello|howdy|howru|yo" : "greetings", etc, etc }

---

In [ ]:
# Create 5 RE patterns

# Dictionary name: "re_patterns"

# ___ Write your code here _____

## Functions: Pattern matching and response retrieval.

---

Using data created above, write programs to detect patterns and retrieve responses along for chatbot. 
 
<span style="color:red; font-size: 15px"> TIP: Here's a nice way to organize next bits of code. 
- Use a cell for each function (makes debugging easier)
- Auxillary functions to detect patterns and generate responses are in the cells above the main.
- The main python loop is the bottom most cell.
- Bespoke pattern matching code that you will write will be above these functions.

---

### 2.3 Function : Detect pattern from input

---

Use regular expressions + dictionary from before to detect a pattern from user input. 

**Function input** : User Input ("Hi, how are you?")  <br>
**Output**         : Response_type ("Greetings") 

---

In [ ]:
## Function name: "detect_pattern"

## Create the detect_pattern function with one argument : user_input
## 1. Loop through each key in the dictionary:
## 2. Write if-else conditions to check for pattern match
##    2.1 For the if condition, use the re.search to check if the pattern exists in user input 
##    2.2 If there is a match, return the response type (e.g., "greetings" or "bye" or "happy")
##    2.3 If none of the conditions match, return "default" which will be the default response of the chatbot
##    Note: use else for last conditon

## TIP: To avoid multiple conditions from matching, use a break statement. 

# Give the function a characteristic name (e.g., detect_pattern) 
# ___ Write your code here _____

### 2.4 Function : Retrieve response

---

Use the output of pattern detection function (above) to trigger a chatbot response. 

In the function, use the "responses" dictionary from above. 


Follow the steps to write function. The function returns a response (e.g., "hello", or "how are you").

**Function input** : User Input ("Hi, how are you?")  <br>
**Detect pattern** : Response_type ("Greetings")  <br>
**Output**         : One of ["hello", "how are you?", etc...]

---

In [ ]:
## Function name: "chatbot_response" 

# Write a function to return a responses by indexing dictionary with response categories.

## Create the chatbot_response function with one argument : user_input
## 1. Call the detect_pattern function and store the response in variable (e.g., detected_type)
## 2. Write if-else conditions to check for pattern match
##    2.1 For the if condition, check if the detected_type is one of the keys of the dictionary
##    2.2 If there is a match, return the response from zipped responses
##    2.3 If none of the conditions, use "default" response

# Give the function a characteristic name (e.g., detect_pattern) 
# ___ Write your code here _____

In [ ]:
## A reminder for what the main loop looked like.
def chat():
    print("Chatbot: Hello! How can I help you today?")
    while True:
        user_input = input("You: ")

        # Generate chatbot response
        bot_response = chatbot_response(user_input)
        print(f"Chatbot: {bot_response}")

# Start chatting with the bot
#chat()

---

<span style="font-family: 'Georgia'; font-size: 15px;">

<span style="color:red;">  Before you proceed .... </span>

- Save the notebook and make a duplicate copy of this notebook 
- Name that copy - chatbot_re_basic_version 
- Now, we will proceed to make more changes to other parts of chatbot code. 

<span>

---

## String substitution


Variable parts can be introduced in a string and substituted with meaningful patterns from user input to make the chatbot's responses more specific. 

This is achieved by string substituitions. Typical substituions can be about user's name, their favourite color, something they mention in the earlier part of the conversation.

Let's look at following example in context of a question & answer chatbot.

### 2.5 Pre-defined templates and responses  <br>

<span style="font-family: 'Georgia'; font-size: 15px;">

The lectures gave us a great starting point, but we're going to start to produce our own independent work now.

In this section, you will further customize chatbot with pre-defined template repsonses and advanced regular expressions.

**Only attempt this after finishing the basic chatbot.**

---

In [ ]:
print('I have a {food_item} and a {drink_item} with me'.format(drink_item='soda', food_item='sandwich'))

# 1. Define necessary variables and complete the following print statement. Uncomment and run it. 

#print('We have {number} {container} containing {qty} gallons of {food_item}'.format())

# 2. Now, store all the variables in a dictionary. Use the dictionary to complete the following print statement. Uncomment and run it. 

#print('We have {number} {container} containing {qty} gallons of {food_item}'.format())


Now, rewrite some of your older responses as template strings and see if you can substitue them with values from user input. 

If you want to attempt a newer one for user name and favourite color, follow the instructions below.

---

In [ ]:
# ___ Write your code here _____

### 2.6 Advanced regular expressions

Let's exploring advanced REs by creating two more patterns to match the name and favourite color of the user. 

As name, and color can be any value, pattern matching needs to be generic and requires using alphabets [a-zA-Z], words [\w] and special symbols [*,+,?].

In [ ]:
# Write a regular expression pattern for a name and favourite color.

# 1. Names typically start with a letter in upper case followed by a sequence of letters in lower case.
# Try writing a few use cases to ensure that the pattern works.

# 2. Colors typically start with a pattern - color is <color_name> or I like <color_name>.
# Try writing a few use cases to ensure that the pattern works.


In [ ]:
# ___ Write your code here _____

# Write few examples of input strings containing names and color to ensure that the pattern works.

#### Integrating name and color into the chatbot 

Lastly, let's integrate the new features into the chatbot.

<span style="color:red; font-size: 15px; position:center "> Be very methodological as you start integrating these new patterns! At every step, pause, write a test case, check for outputs and then proceed! </span>  


##### 1. Expanding pattern detection (detect_pattern function)
    - Create user_state as a global variable dictionary above the detect_pattern function.
    - Along with other if-else conditions, add two regular expression patterns for finding name and favourite color.
    - If there is a match with name or color
        - Get the result of the match using result.group()
        - Store the match in the corresponding key of the user_state dictionary.
        - return "name_response" or "color_response" according to the condition

##### 2. Expanding responses
    - Add the two responses for the new categories - name and color. 
    - These responses should be in the format - 'hello {name}, how are you today?' or 'Great to know that your favourite color is {color}'

##### 3. Modifying chatbot responses
    - In chatbot_response, following the function call to detect_pattern, create 2 variables, name and color.
    - Store the values from user_state in name and color.
    - Along with other if-else conditions, add two conditions for name_response and color_response
    - If there is a match,
        - Retrieve the match from responses dictionary and store it as "response"
        - Use string substitution operation (response.format) with the name or color variable.

#### At this point, verify that the code works correctly!!


<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 2 </span> </div>
<hr style="border: 2px solid green;">

<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 3 </span>

**File handling**
    
In this part, you will create and load a JSON file to drive the chatbot's responses. At the end of this step,
we will be truly close to achieving full independence between data and process/program. 
 
Pre-readings: Lecture material of **Week 7 and 8** on **File handling**. Please read before before attempting.
    
<span style="color:red"> *Before you start, complete all the previous parts (Parts 1 and 2).*

**Learning outcomes**
- Reading data from JSON file
- Populating chatbot knowledge structures (patterns,intents,responses)
- Integrating file-based data into chatbot logic

Complete this activity before proceeding to the next parts of the chatbot.

</span>

---

### 3.1 Reading chatbot intents from a JSON file

Apply the file handling concepts and pandas to read JSON file (intents.json) and store its content.


In [ ]:
### Write the function - read_json()...


In [ ]:
### Call the function and store intents in intents_json

## Write code here...

In [ ]:
### Print print only the patterns of intents_json to see 
### their content as that's directly relevant for next step.


--- 

**Here are some questions to help you guide your decision-making for next steps**:

- From some of the intent/response pairs , what do the contents seem to be about? 
- Are all patterns strings? Are some regular expressions? 
- What is the best format to convert them to? 

--- 

---

### 3.2 Populating chatbot knowledge structures (patterns,intents,responses)

The logical choice for creating these mappings is to use dictionaries. 

**Remember**: While constructing a dictionary, you want the the **key-value** to be same data type. But, patterns are regular expressions, intents are strings, and responses are lists or strings. 

In the following cells, practice type conversions from strings to regular expressions, to construct the right dictionaries for mappings. 

Pay attention to the different formats while constructing these dictionaries.

---

### Strings -> regex

In [ ]:
# To convert patterns from strings to REs, we have to look 
# at each pattern and concatenate them with a pipe symbol.

# Loop thorugh patterns list and tags.
 # For each pattern in list, add an "r" symbol
 # Create a dictionary with pattern and tags

patterns = ["hello","hi","hiya"]

r"hello|hi|hiya"

r"|".join(patterns)


### Dictionary 1: Patterns -> intent 

In [ ]:
## Create an empty dictionary variable rgx2int for mapping from pattern to tags/intents

## Write code here....

### Dictionary 2: intent -> responses


TIP: For any feature, write this as a function. (e.g. store_responses)

In [ ]:
## Create an empty dictionary int2res for mapping from intent to responses.

# Loop through the list of JSON objects:
 # For each object in list, get responses and tag
 # Add to the dictionary: tags as keys and responses as values

### 3.3 Integrating file-based data into chatbot logic

Follow the code templates provided below to understand how files can be integrated with the Chatbot logic. 

*Fill in appropriate code segments.*


In [ ]:
## A code template for storing responses from JSON file..

# def load_files():
#    pd.read_json('intents.json')
#    Type your code here.

In [ ]:
## A template for what the response generation code will look like..

def generate_response(user_input):
    
    rgx2int, int2res = load_files()
    
    # Checks for user input in rgx2int
    # Uses tag of rgx2int in int2res to get output
    
    # Print match and return true
    # or return False 

In [ ]:
## A reminder for what the main loop code will look like..

# Function to get user input and respond
def chatbot():

    # User welcome and termination instruction.
    print("Welcome! Press bye to exit")

    # Main loop:
    while True:
        
        user_input = input("You: ")
                
        if user_input == "bye":
            print("Chatbot: Thank you for your interaction")
            break
        
        response_found = generate_response()
                
        if response_found == False:
            print("I'm sorry, could you rephrase that again")
        
# Run the chatbot
chatbot()


<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 3 </span> </div>
<hr style="border: 2px solid green;">

<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 4 </span>        
**NLTK and Word substitutions**
    
<span style="color:red; font-size: 15px">**Note:**</span> *You will notice that this part is
bit less structured than previous parts. As you proceed to finalise 
the details of your chatbot for submission, 
reflect and understand the
code structure from the previous parts. Use this observation to 
meaningfully extend the chatbot
new pre-processing features.*



Before matching patterns, the chatbot should map certain words to alternatives that are easier for the chatbot to handle.

In this part, you will implement stemmers and lemmatizers that perform these substitutions on the user’s input before further processing.
    
Please get yourself familiar with **Week 9 and 10** of lecture material on **NLTK and text pre-processing** before attempting this. 

<span style="color:red"> *Before you start this, please ensure that you complete all the previous parts (Parts 1, 2 and 3).* </span>

**Learning outcomes**
- Tokenizing and preprocessing user input (punctuation, and stopwords)
- Word substitutitons (stemming, and lemmatization)
- Integrating preprocessing into chatbot pipeline

This is the last part of the chatbot. 

</span>

--- 


### Word substitutions

Before matching patterns, the chatbot should map certain words to alternatives that are easier for the chatbot to handle. For example:

    discussed → discuss 
    running → run 


<span style="font-family: 'Georgia'; font-size: 18px; "> Identify stems and root words </span>
    
In the following conversational snippet, identify how stemmers and lemmatizers are used to identify word variations.

-> Stemmer is used to identify "ing" forms of a verb (e.g., running and run) <br>
-> Lemmatizer is used recognize words with same root word (e.g., better and good)

    
The chatbot you build should demonstrate a behavior similar to this.

### Sample conversation snippet:

> User: I was running late to the NLP class today.

> Chatbot: Oh no! Were you feeling stressed?

> User: Yes, I was worrying about missing the lecture on stemmers and lemmatizers.

> Chatbot: Don’t worry, they were just discussing the stemmers and lemmatizers.

> User: Can you explain what they discussed?

> Chatbot: They explained how stemmers just cut off word endings, while lemmatizers consider the context.

> User: Oh, so like running becomes run? 

> Chatbot: Yes, running becomes run (**Stemmer applied**)

> User: What does better become?

> Chatbot: "Running" becomes "run" in both, but “better” becomes “good” only in lemmatization (**Lemmatizer applied**)

> User: That makes sense. So, choosing between them depends on the context!

> Chatbot: Yes. If speed is important, use stemming. If accuracy matters, use lemmatization.

> User: Thanks

> Chatbot: No worries! Have a great day!


In [ ]:
# ___ Write your code here _____


## Further steps:
    
### Synonym Handling 

Your chatbot can be further improved by handling synonyms through substitution. 
    
Create a rule file with sets of synonyms that the chatbot should treat as equivalent for 
pattern matching purposes. For example, the words "sad", "unhappy", and "depressed" are considered synonymous.

You will need to implement synonym substitution, where the input is normalized before attempting to match any   patterns. For example, if a user says, "I am unhappy," the chatbot should recognize that "unhappy" is  
synonymous with "sad."



<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 4 </span> </div>
<hr style="border: 2px solid green;">